# Decisions API: When Your App Needs a Decision

Compare typed decisions with generated JSON on the same routing task, measuring speed, estimated cost and correctness.

Decisions returns probabilities, fixed choices or rubric scores. It does not generate an explanation or request a tool call. **The endpoint has this output contract; GPT-6 Luna can still generate text through Responses.**

Use Decisions for repeated classification, filtering, routing or prioritisation. Use Responses for writing, extracting arbitrary fields, explanations and tool calls with arguments. Use ordinary code when an exact rule is sufficient.

## 1. Setup

Use Python 3.10+ and an OpenAI API key. Decisions is in public beta and currently supports `gpt-6-luna`. Examples use the native Python SDK, version 3.26.0. [Official guide](https://developers.openai.com/api/docs/guides/decisions)

The introduction makes three paid requests. The optional benchmark adds 40 requests. All examples use fictional support tickets.

In [ ]:
%pip install openai==3.26.0 pandas matplotlib --upgrade --quiet

In [ ]:
import json
import os
import random
from getpass import getpass
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from openai import AsyncOpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"], timeout=30, max_retries=0)
MODEL = "gpt-6-luna"  # Same model on both endpoints.

### Jev and the interest in decision models

TypeSafe AI launched **Jev** on 15 September 2026 with a simple pitch: fast, typed decisions with probabilities, instead of generated text, and pricing based on input tokens. Community projects such as Jared Palmer's **Kev**, explicitly described as Jev-like, show how the idea spread to developers. This is context for its early visibility, not an adoption ranking.

TypeSafe calls its training **Reinforcement Learning for Calibrated Decisions (RLCD)**. That is a claim about Jev. OpenAI's public Decisions documentation does not disclose the same training method. Here we use OpenAI's API to explore the developer pattern.

[Jev launch](https://typesafe.ai/blog/introducing-system-one-models-and-jev) · [Kev models](https://huggingface.co/collections/jaredpalmer/kev) · [TypeSafe's calibration primer](https://docs.typesafe.ai/introduction/machine-learning-primer)

## 2. Predicate, Choice and Score

| Type | Question | Returned meaning |
| --- | --- | --- |
| `predicate` | Is a workaround explicitly reported? | Probability that the condition is true, from 0 to 1 |
| `choice` | Which department handles this ticket? | One supplied category, plus option probabilities and a separate confidence field |
| `score` | How blocked is the customer? | Probability-weighted average of ordered level indices, starting at 0 |

A score of 1.1 on levels 0, 1 and 2 is an average, not a new category or a probability. Choice has no ordering. Predicate is a probability, not an automatic Boolean.

Ask **independent questions together** about shared input. If a question depends on a previous answer, make a separate request after receiving that answer. [Multiple questions](https://developers.openai.com/api/docs/guides/decisions#ask-multiple-questions)

In [ ]:
TICKET = "CSV export fails in Safari, but exporting from Chrome works."
CHOICES = [
    {"value": "billing", "description": "Payments, invoices or refunds."},
    {"value": "technical", "description": "Software errors or trouble using the product."},
    {"value": "shipping", "description": "Delivery, parcels or tracking."},
    {"value": "other", "description": "Anything outside these categories, including feature suggestions."},
]
ROUTE = {"type": "choice", "name": "department",
         "instructions": "Route the support request to the department that handles it.",
         "choices": CHOICES}
QUESTIONS = [
    {"type": "predicate", "name": "workaround",
     "instructions": "Does the ticket explicitly report an alternative way to complete the task?"},
    ROUTE,
    {"type": "score", "name": "blockage",
     "instructions": "Rate the reported loss of functionality, using only the ticket.",
     "levels": [
         {"label": "Cosmetic", "description": "Appearance issue only; the task still works."},
         {"label": "Workaround", "description": "The task fails in one way but another way works."},
         {"label": "Blocked", "description": "The task cannot be completed and no workaround is reported."},
     ]},
]
decision = await client.decisions.create(model=MODEL, input=TICKET, questions=QUESTIONS)
for answer in decision.answers:
    if answer.type == "refusal":
        print(f"{answer.name}: refused; send to review.")
    else:
        display(answer.model_dump())
display(decision.usage.model_dump())

### Check the answers against ground truth

Chrome completes the export, so our labels are **workaround = True**, **department = technical**, and **blockage level = 1 (Workaround)**. For this demonstration, Predicate becomes a Boolean at 0.5 and we check Score's most likely level. We inspect its weighted average separately below.

A valid type or a high probability does not guarantee a correct answer. Keep failed checks visible. This single ticket also cannot establish calibration.

In [ ]:
answers = {answer.name: answer for answer in decision.answers}
workaround, route, score = [answers[name] for name in ["workaround", "department", "blockage"]]
predictions = {
    "workaround": workaround.probability >= 0.5 if workaround.type == "predicate" else None,
    "department": route.choice if route.type == "choice" else None,
    "blockage": max(score.probabilities, key=lambda p: p.probability).value if score.type == "score" else None,
}
GROUND_TRUTH = {"workaround": True, "department": "technical", "blockage": 1}
demo_checks = pd.DataFrame([
    {"question": name, "ground_truth": expected, "prediction": predictions[name],
     "correct": predictions[name] == expected}
    for name, expected in GROUND_TRUTH.items()
])
display(demo_checks)
if not demo_checks["correct"].all():
    print("At least one decision disagrees with the labels. Review the failure before automating.")

## 3. How a decision becomes an action

**Evidence + question → model evaluation → typed answer → application policy.** The API returns probabilities rather than generated prose. Your code selects thresholds and performs any action.

A classification head producing logits followed by softmax is a useful *conceptual* way to think about a classifier. Softmax normalises logits into probabilities; log probabilities are the logarithms of those probabilities. This API exposes probabilities, not raw logits or token log probabilities. The public Decisions guide does **not** specify its head architecture, training objective or a policy-reward training loop, so we should not teach those as implementation facts. Training a model with rewards is also separate from applying your application's policy at runtime.

Calibration means predicted probabilities agree with observed frequencies on representative labelled data. A probability field alone does not prove calibration on your task. Set thresholds using held-out labels and the costs of mistakes. The API's separate `confidence` field is not interchangeable with an option's probability. [Interpret answers](https://developers.openai.com/api/docs/guides/decisions#interpret-the-answers)

In [ ]:
# Illustrative threshold only. Tune it using your application's labelled data.
answers = {answer.name: answer for answer in decision.answers}
route = answers["department"]
if route.type == "choice":
    chosen_probability = next(p.probability for p in route.probabilities if p.value == route.choice)
    queue = route.choice if chosen_probability >= 0.90 and route.choice != "other" else "review"
    print(f"P(chosen option)={chosen_probability:.3f}; confidence={route.confidence:.3f}; queue={queue}")
else:
    print("No route returned; send to review.")

score = answers["blockage"]
if score.type == "score":
    reconstructed = sum(p.value * p.probability for p in score.probabilities)
    print(f"Returned score={score.score:.3f}; weighted level indices={reconstructed:.3f}")

## 4. Compare speed and cost fairly

We compare **one department decision** on each endpoint, using the same model, ticket, options and routing criteria. Responses generates only a small JSON object with an enum. It uses no reasoning tokens or tools, so we avoid inflating the baseline with an essay or a larger model.

The request formats differ because the APIs have different contracts. We measure local end-to-end latency, including network time, and calculate an **estimate from returned usage**, not an invoice. The first pair warms both paths; it is excluded from the repeated benchmark.

Rates checked on 8 October 2026, USD per 1M tokens:

| Endpoint | Fresh input | Cached input | Cache writes | Output |
| --- | ---: | ---: | ---: | ---: |
| Decisions | $0.10 | No separate cache charge | No separate cache charge | $0 |
| Responses, GPT-6 Luna Standard | $0.10 | $0.01 | $0.125 | $0.50 |

Estimates assume short text inputs and standard processing, without regional premiums, tax or other services. Responses can have cheaper cached input; Decisions is not guaranteed to be cheaper for every workload. [Decisions pricing](https://developers.openai.com/api/docs/guides/decisions#pricing-and-availability) · [Responses pricing](https://developers.openai.com/api/docs/pricing)

In [ ]:
RATES = {"input": 0.10, "cached": 0.01, "cache_write": 0.125, "output": 0.50}
FORMAT = {"type": "json_schema", "name": "department_route", "strict": True,
          "schema": {"type": "object", "properties": {
              "department": {"type": "string", "enum": [c["value"] for c in CHOICES]}},
              "required": ["department"], "additionalProperties": False}}


def estimated_cost(endpoint, usage):
    """Short-context Standard USD estimate, using returned token counts."""
    details = usage.get("input_tokens_details", {})
    cached, written = details.get("cached_tokens", 0), details.get("cache_write_tokens", 0)
    if endpoint == "decisions":
        return usage["input_tokens"] * RATES["input"] / 1_000_000
    fresh = usage["input_tokens"] - cached - written
    if fresh < 0:
        raise ValueError("Unexpected cache accounting; review the current usage schema.")
    return (fresh * RATES["input"] + cached * RATES["cached"]
            + written * RATES["cache_write"] + usage["output_tokens"] * RATES["output"]) / 1_000_000

In [ ]:
async def measure(endpoint, ticket, expected):
    """Measure one routing call. Retain failures instead of dropping them."""
    started = perf_counter()
    usage, selected, error = None, None, ""
    try:
        if endpoint == "decisions":
            result = await client.decisions.create(model=MODEL, input=ticket, questions=[ROUTE])
            answer = result.answers[0]
            selected = answer.choice if answer.type == "choice" else None
        else:
            result = await client.responses.create(
                model=MODEL, input=ticket,
                instructions=ROUTE["instructions"] + "\nAllowed categories:\n" + json.dumps(CHOICES),
                reasoning={"effort": "none"}, max_output_tokens=64,
                text={"format": FORMAT}, store=False, service_tier="default",
            )
            if result.status == "completed" and result.output_text:
                selected = json.loads(result.output_text)["department"]
        elapsed = perf_counter() - started
        usage = result.usage.model_dump()
        if selected is None:
            error = "refusal_or_no_completed_answer"
    except Exception as exc:
        elapsed = perf_counter() - started
        error = type(exc).__name__
    details = (usage or {}).get("input_tokens_details", {})
    return {"endpoint": endpoint, "choice": selected, "expected": expected,
            "correct": selected == expected, "latency_ms": elapsed * 1000,
            "input_tokens": usage["input_tokens"] if usage else None,
            "output_tokens": usage["output_tokens"] if usage else None,
            "cached_tokens": details.get("cached_tokens", 0) if usage else None,
            "cache_write_tokens": details.get("cache_write_tokens", 0) if usage else None,
            "estimated_usd": estimated_cost(endpoint, usage) if usage else None, "error": error}

# These two requests are the warm-up pair, not a benchmark conclusion.
single_pair = [await measure(endpoint, TICKET, "technical") for endpoint in ["decisions", "responses"]]
display(pd.DataFrame(single_pair))

## 5. Repeat on labelled tickets

Set `RUN_BENCHMARK = True` for 20 pairs, or 40 paid requests. Calls run sequentially and the endpoint order is randomised within each pair. This measures request latency, not maximum throughput.

Two repeats of ten tickets show timing variation. They do not create twenty independent accuracy examples. Use a larger representative holdout to assess quality, calibration or production tail latency. Errors count against correctness and are reported separately; cost is known only where usage was returned.

In [ ]:
CASES = [
    ("My card was charged twice for one order.", "billing"),
    ("The invoice total is different from the amount I paid.", "billing"),
    ("Saving a project shows an error and discards my edits.", "technical"),
    ("The password-reset link returns a 500 error.", "technical"),
    ("My parcel has not arrived, although delivery was due yesterday.", "shipping"),
    ("Tracking says my parcel was delivered in a different town.", "shipping"),
    ("Are you hiring interns?", "other"),
    ("Could you add a dark mode in a future release?", "other"),
    ("Please refund the duplicate payment for my order.", "billing"),
    ("The export button crashes the browser.", "technical"),
]
RUN_BENCHMARK = False
REPEATS = 2
benchmark_results = None
if RUN_BENCHMARK:
    rng = random.Random(7)
    schedule = list(range(len(CASES))) * REPEATS
    rng.shuffle(schedule)
    runs = []
    for pair_id, case_id in enumerate(schedule):
        ticket, expected = CASES[case_id]
        endpoints = ["decisions", "responses"]
        rng.shuffle(endpoints)
        for endpoint in endpoints:
            row = await measure(endpoint, ticket, expected)
            runs.append({"pair_id": pair_id, "case_id": case_id, **row})
    benchmark_results = pd.DataFrame(runs)
else:
    print("Set RUN_BENCHMARK = True to measure 20 pairs.")

In [ ]:
def benchmark_report(results):
    rows = []
    for endpoint, group in results.groupby("endpoint", sort=False):
        valid = group[group["error"].eq("")]
        rows.append({"endpoint": endpoint, "calls": len(group),
            "correctness": group["correct"].mean(), "errors": group["error"].ne("").sum(),
            "p50_ms": valid["latency_ms"].median(), "p95_ms": valid["latency_ms"].quantile(0.95),
            "known_usage_usd": group["estimated_usd"].sum(min_count=1),
            "mean_usd_per_1000_calls": valid["estimated_usd"].mean() * 1000,
            "mean_input_tokens": valid["input_tokens"].mean(),
            "mean_output_tokens": valid["output_tokens"].mean(),
            "mean_cached_tokens": valid["cached_tokens"].mean()})
    return pd.DataFrame(rows).set_index("endpoint")

if benchmark_results is not None:
    report = benchmark_report(benchmark_results)
    display(report.round(6))
    if report["errors"].sum() == 0:
        print(f"Measured p50 latency ratio, Responses / Decisions: {report.loc['responses', 'p50_ms'] / report.loc['decisions', 'p50_ms']:.2f}x")
        print(f"Estimated mean cost ratio, Responses / Decisions: {report.loc['responses', 'mean_usd_per_1000_calls'] / report.loc['decisions', 'mean_usd_per_1000_calls']:.2f}x")
    else:
        display(benchmark_results.loc[benchmark_results["error"].ne("")])

In [ ]:
if benchmark_results is not None:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    labels = ["Decisions", "Responses"]
    ordered = report.reindex(["decisions", "responses"])
    for axis, column, title, unit in [
        (axes[0], "p50_ms", "Median request latency", "Milliseconds"),
        (axes[1], "mean_usd_per_1000_calls", "Estimated cost per 1,000 calls", "USD"),
    ]:
        axis.bar(labels, ordered[column], color=["#E91D63", "#585858"])
        axis.set(title=title, ylabel=unit)
        axis.spines[["top", "right"]].set_visible(False)
    fig.suptitle("This local routing benchmark, not a universal speed or cost claim", fontsize=10)
    fig.tight_layout()
    plt.show()

## 6. What the result means for a developer

First check correctness and errors. Then compare median latency, the p95 estimate and known token costs. The guide advertises about 10x faster decisions, but your minimal JSON baseline, network, caching and workload can produce a different ratio. A lower output-token count is not by itself a complete cost comparison.

Use a decision endpoint when the allowed answer space covers the task and repeated decisions justify it. Add an `other` option, handle refusal and send uncertain inputs to review. Your application still owns permissions and business rules.

Use Responses when you need a written answer, extracted data with your own schema, tool arguments or a workflow that gathers missing evidence. A routing label can choose a handler; it cannot fetch an order, invent missing facts or write the customer's reply. A useful combination is **Decisions for routing, Responses for the reply, code for the actual action**.

## Summary

- Match the output contract to the task: probability, fixed category or ordered rubric score.
- Batch independent questions about the same input; separate dependent steps.
- Keep model and routing criteria comparable when measuring speed and cost. Use returned usage and check quality too.
- Validate calibration and thresholds on labelled data. The API contract does not reveal a specific head or reward-training pipeline.

[Decisions Playground](https://platform.openai.com/decisions?lang=python) · [Guide](https://developers.openai.com/api/docs/guides/decisions) · [Python API reference](https://developers.openai.com/api/reference/python/resources/decisions/methods/create) · [GPT-6 Luna](https://developers.openai.com/api/docs/models/gpt-6-luna)